In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
AMI_DIR = os.path.join(PROJECT_DIR, "data", "raw", "ami")

os.makedirs(AMI_DIR, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("AMI_DIR:", AMI_DIR)

PROJECT_DIR: /content/drive/MyDrive/MTechIndProj/MoM_Project
AMI_DIR: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami


In [3]:
!pip -q install -U huggingface_hub requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
import requests

url = "https://groups.inf.ed.ac.uk/ami/download/"

response = requests.get(url, timeout=30)

print("Status:", response.status_code)
print("Page size:", len(response.content), "bytes")

Status: 200
Page size: 15189 bytes


In [5]:
import requests
from bs4 import BeautifulSoup

AMI_URL = "https://groups.inf.ed.ac.uk/ami/download/"

html = requests.get(AMI_URL, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

links = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(" ", strip=True)

    if "ES200" in href or "ES200" in text:
        links.append((text, href))

print("AMI meeting-related links found:", len(links))

for text, href in links[:30]:
    print(text, "->", href)

AMI meeting-related links found: 0


In [6]:
import requests
from bs4 import BeautifulSoup

AMI_URL = "https://groups.inf.ed.ac.uk/ami/download/"

response = requests.get(AMI_URL, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

forms = soup.find_all("form")

print("Number of forms:", len(forms))

for i, form in enumerate(forms):
    print("\nFORM", i)
    print("Action:", form.get("action"))
    print("Method:", form.get("method"))

    for element in form.find_all(["input", "select", "option"]):
        print(
            element.name,
            element.get("name"),
            element.get("value"),
            element.get_text(" ", strip=True)
        )

Number of forms: 2

FORM 0
Action: None
Method: None

FORM 1
Action: https://homepages.inf.ed.ac.uk/cgi/simonk/AMI/estimate.cgi
Method: POST
select sessionID None ES2002 ES2003 ES2004 ES2005 ES2006 ES2007 ES2008 ES2009 ES2010 ES2011 ES2012 ES2013 ES2014 ES2015 ES2016
option None  
option None ES2002 ES2002
option None ES2003 ES2003
option None ES2004 ES2004
option None ES2005 ES2005
option None ES2006 ES2006
option None ES2007 ES2007
option None ES2008 ES2008
option None ES2009 ES2009
option None ES2010 ES2010
option None ES2011 ES2011
option None ES2012 ES2012
option None ES2013 ES2013
option None ES2014 ES2014
option None ES2015 ES2015
option None ES2016 ES2016
input ES a 
input ES b 
input ES c 
input ES d 
select sessionID None IS1000 IS1001 IS1002 IS1003 IS1004 IS1005 IS1006 IS1007 IS1008 IS1009
option None  
option None IS1000 IS1000
option None IS1001 IS1001
option None IS1002 IS1002
option None IS1003 IS1003
option None IS1004 IS1004
option None IS1005 IS1005
option None IS1006

In [7]:
import requests

AMI_ESTIMATE_URL = "https://homepages.inf.ed.ac.uk/cgi/simonk/AMI/estimate.cgi"

payload = [
    ("sessionID", "ES2004"),
    ("ES", "a"),
    ("mediaType", "DivXvideos"),
    ("mediaType", "Mix-Headset"),
    ("mediaType", "slides"),
    ("mediaType", "smi"),
]

response = requests.post(
    AMI_ESTIMATE_URL,
    data=payload,
    timeout=60
)

print("Status:", response.status_code)
print("Response length:", len(response.text))
print(response.text[:3000])

Status: 200
Response length: 31146
    
    <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd"><html xmlns="http://www.w3.org/1999/xhtml" xml:lang="fr" lang="fr">
    <head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
    <title> AMI samples / builds </title>
    <!-- <link rel="stylesheet" href="/ferret.css" type="text/css"> -->
    <style type="text/css">    
    body,td,div,.p{font-family:arial,sans-serif;font-size:10pt; }
    th.blue{color:black; font-size:10pt; font-family:verdana,sans-serif;
    background-color:#99CCFF;44aaCC; f2f2f4;#66CCFF;
    border-width:1; border-color:black; vertical-align:center}
    td, table{	list-style-type: none;	padding: 1px;}
</style>
    <script>
function getText(elementname,name,paste,prerequisites) {
  element = document.getElementById(elementname);
  navigator.clipboard.writeText(element.value);
  alertstr = name+' is now in your clipboard. Paste your clipbo

In [11]:
from bs4 import BeautifulSoup

result_soup = BeautifulSoup(response.text, "html.parser")

textarea = result_soup.find("textarea", id="wgetsh")

print("Textarea found:", textarea is not None)

if textarea is not None:
    script = textarea.get_text()
    print("Script length:", len(script))
    print("\nFirst 5000 characters:\n")
    print(script[:5000])
else:
    print("wgetsh textarea was not found.")

    print("\nAvailable textarea IDs:")
    for t in result_soup.find_all("textarea"):
        print(t.get("id"))

Textarea found: True
Script length: 0

First 5000 characters:




In [13]:
import re
import html

match = re.search(
    r"fillText\('wgetsh','(.*?)'\)",
    response.text,
    re.DOTALL
)

print("Match found:", match is not None)

if match:
    script = match.group(1)
    script = script.replace("\\n", "\n")
    script = html.unescape(script)

    print("Script length:", len(script))
    print("\nGenerated download script:\n")
    print(script)
else:
    print("Could not find the generated wget script.")

Match found: True
Script length: 9810

Generated download script:

#!/bin/sh
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.PreferredOverview.avi
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.Closeup1.avi
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.Closeup2.avi
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.Closeup3.avi
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.Closeup4.avi
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.Corner.avi
wget    -P amicorpus/ES2004a/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004a/video/ES2004a.Overhead.avi
wget    -P am

In [14]:
import os
import requests
from urllib.parse import urlparse

MEETING_ID = "ES2004a"

MEETING_DIR = os.path.join(
    AMI_DIR,
    MEETING_ID
)

VIDEO_DIR = os.path.join(MEETING_DIR, "video")
AUDIO_DIR = os.path.join(MEETING_DIR, "audio")
SLIDES_DIR = os.path.join(MEETING_DIR, "slides")

for directory in [VIDEO_DIR, AUDIO_DIR, SLIDES_DIR]:
    os.makedirs(directory, exist_ok=True)

BASE_URL = "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/"

files_to_download = [
    (
        f"{MEETING_ID}/video/{MEETING_ID}.PreferredOverview.avi",
        VIDEO_DIR
    ),
    (
        f"{MEETING_ID}/audio/{MEETING_ID}.Mix-Headset.wav",
        AUDIO_DIR
    ),
]

# Add slide files from the generated AMI script
for line in script.splitlines():
    if f"{MEETING_ID}/slides/" in line and line.startswith("wget"):
        url = line.split()[-1]
        filename = os.path.basename(urlparse(url).path)

        files_to_download.append(
            (f"{MEETING_ID}/slides/{filename}", SLIDES_DIR)
        )

print("Files selected:", len(files_to_download))

Files selected: 64


In [15]:
for relative_path, output_dir in files_to_download:

    url = BASE_URL + relative_path
    filename = os.path.basename(relative_path)
    output_path = os.path.join(output_dir, filename)

    if os.path.exists(output_path):
        print("Already exists:", filename)
        continue

    print("\nDownloading:")
    print(filename)

    try:
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()

        total = int(r.headers.get("content-length", 0))
        downloaded = 0

        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

        print("✓ Downloaded:", round(downloaded / (1024**2), 2), "MB")

    except Exception as e:
        print("✗ Failed:", filename)
        print("  Error:", e)


Downloading:
ES2004a.PreferredOverview.avi
✗ Failed: ES2004a.PreferredOverview.avi
  Error: 403 Client Error: Forbidden for url: https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004a/video/ES2004a.PreferredOverview.avi

Downloading:
ES2004a.Mix-Headset.wav
✓ Downloaded: 32.02 MB

Downloading:
ES2004a.586.44__693.07.jpg
✓ Downloaded: 0.16 MB

Downloading:
ES2004a.1004.86__1019.17.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.1019.17__1032.91.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.1032.91__1066.12.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.111.35__138.24.jpg
✓ Downloaded: 0.16 MB

Downloading:
ES2004a.111.35__138.24.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.138.24__150.59.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.138.24__162.30.jpg
✓ Downloaded: 0.15 MB

Downloading:
ES2004a.150.59__162.30.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.162.30__174.27.txt
✓ Downloaded: 0.0 MB

Downloading:
ES2004a.162.30__191.57.jpg
✓ Downloaded: 0.17 MB

Downloading:
E

In [16]:
for root, dirs, files in os.walk(MEETING_DIR):
    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"{size_mb:8.2f} MB  {path}")

   32.02 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
    0.16 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.586.44__693.07.jpg
    0.00 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.1004.86__1019.17.txt
    0.00 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.1019.17__1032.91.txt
    0.00 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.1032.91__1066.12.txt
    0.16 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.111.35__138.24.jpg
    0.00 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.111.35__138.24.txt
    0.00 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.138.24__150.59.txt
    0.15 MB  /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/

In [17]:
import os
from pathlib import Path

for resource in ["video", "audio", "slides"]:
    path = Path(AMI_DIR) / "ES2004a" / resource

    files = list(path.glob("*"))

    total_size = sum(f.stat().st_size for f in files)

    print(f"\n{resource.upper()}")
    print("Files:", len(files))
    print("Size:", round(total_size / (1024**2), 2), "MB")

    for f in files[:5]:
        print(" -", f.name)


VIDEO
Files: 0
Size: 0.0 MB

AUDIO
Files: 1
Size: 32.02 MB
 - ES2004a.Mix-Headset.wav

SLIDES
Files: 62
Size: 1.65 MB
 - ES2004a.586.44__693.07.jpg
 - ES2004a.1004.86__1019.17.txt
 - ES2004a.1019.17__1032.91.txt
 - ES2004a.1032.91__1066.12.txt
 - ES2004a.111.35__138.24.jpg


In [18]:
video_url = (
    "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/"
    "amicorpus/ES2004a/video/ES2004a.PreferredOverview.avi"
)

r = requests.head(video_url, timeout=30)

print("Status:", r.status_code)
print("Content-Type:", r.headers.get("content-type"))
print("Content-Length:", r.headers.get("content-length"))

Status: 403
Content-Type: text/html; charset=iso-8859-1
Content-Length: None


In [19]:
video_url = (
    "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/"
    "amicorpus/ES2004a/video/ES2004a.PreferredOverview.avi"
)

headers = {
    "User-Agent": "Mozilla/5.0"
}

try:
    r = requests.get(
        video_url,
        headers=headers,
        stream=True,
        timeout=60
    )

    print("Status:", r.status_code)
    print("Content-Type:", r.headers.get("content-type"))
    print("Content-Length:", r.headers.get("content-length"))

    r.close()

except Exception as e:
    print("Error:", repr(e))

Status: 403
Content-Type: text/html; charset=iso-8859-1
Content-Length: 269


In [20]:
import os

video_dir = os.path.join(
    AMI_DIR,
    "ES2004a",
    "video"
)

os.makedirs(video_dir, exist_ok=True)

video_url = (
    "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/"
    "amicorpus/ES2004a/video/ES2004a.PreferredOverview.avi"
)

!wget -O "{video_dir}/ES2004a.PreferredOverview.avi" "{video_url}"

--2026-08-26 07:38:52--  https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004a/video/ES2004a.PreferredOverview.avi
Resolving groups.inf.ed.ac.uk (groups.inf.ed.ac.uk)... 129.215.202.26
Connecting to groups.inf.ed.ac.uk (groups.inf.ed.ac.uk)|129.215.202.26|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-08-26 07:38:53 ERROR 403: Forbidden.



In [21]:
import requests

url = "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2004a/video/"

r = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status:", r.status_code)
print(r.text[:3000])

Status: 200
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2 Final//EN">
<html>
 <head>
  <title>Index of /ami/AMICorpusMirror/amicorpus/ES2004a/video</title>
 </head>
 <body>
<h1>Index of /ami/AMICorpusMirror/amicorpus/ES2004a/video</h1>
  <table>
   <tr><th valign="top"><img src="/icons/blank.gif" alt="[ICO]"></th><th><a href="?C=N;O=D">Name</a></th><th><a href="?C=M;O=A">Last modified</a></th><th><a href="?C=S;O=A">Size</a></th><th><a href="?C=D;O=A">Description</a></th></tr>
   <tr><th colspan="5"><hr></th></tr>
<tr><td valign="top"><img src="/icons/image2.gif" alt="[IMG]"></td><td><a href="ES2004a.Closeup1-face.jpeg">ES2004a.Closeup1-face.jpeg</a></td><td align="right">2005-03-29 08:54  </td><td align="right">2.1K</td><td>&nbsp;</td></tr>
<tr><td valign="top"><img src="/icons/movie.gif" alt="[VID]"></td><td><a href="ES2004a.Closeup1.avi">ES2004a.Closeup1.avi</a></td><td align="right">2006-05-15 15:57  </td><td align="right"> 28M</td><td>&nbsp;</td></tr>
<tr><td valign="top"><img src="/

In [22]:
video_url = (
    "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/"
    "amicorpus/ES2004a/video/ES2004a.Closeup1.avi"
)

video_path = os.path.join(
    AMI_DIR,
    "ES2004a",
    "video",
    "ES2004a.Closeup1.avi"
)

r = requests.get(
    video_url,
    headers={"User-Agent": "Mozilla/5.0"},
    stream=True,
    timeout=120
)

print("Status:", r.status_code)
print("Content-Type:", r.headers.get("content-type"))

if r.status_code == 200:
    with open(video_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print("Downloaded:", video_path)
    print(
        "Size:",
        round(os.path.getsize(video_path) / (1024**2), 2),
        "MB"
    )
else:
    print("Download failed.")

Status: 200
Content-Type: video/x-msvideo
Downloaded: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/video/ES2004a.Closeup1.avi
Size: 27.88 MB


In [23]:
import requests
from bs4 import BeautifulSoup

download_page = "https://groups.inf.ed.ac.uk/ami/download/"

html = requests.get(download_page, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]

    if any(word in (text + " " + href).lower()
           for word in ["annotation", "annot", "manual"]):
        print(text, "->", href)

Annotation -> /ami/corpus/annotation.shtml
Annotations present for each meeting -> /ami/corpus/annotationpresent.shtml
Guidelines for annotators -> /ami/corpus/guidelines.shtml
AMI manual annotations v1.6.2 -> https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/ami_public_manual_1.6.2.zip
AMI automatic annotations v1.5.1 -> https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/ami_public_auto_1.5.1.zip
annotations -> https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/dome_annotations_M1.csv
dataset -> https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/dome_dataset_M1.csv
documentation -> https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/DOME_documentation.pdf
Social role annotation -> https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/SocialRoleAnnotation.tar.gz


In [24]:
import os
import requests

ANNOTATION_URL = (
    "https://groups.inf.ed.ac.uk/ami/"
    "AMICorpusAnnotations/ami_public_manual_1.6.2.zip"
)

ANNOTATION_ZIP = os.path.join(
    AMI_DIR,
    "ami_public_manual_1.6.2.zip"
)

print("Downloading AMI manual annotations...")
print("Destination:", ANNOTATION_ZIP)

r = requests.get(
    ANNOTATION_URL,
    stream=True,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=120
)

print("Status:", r.status_code)

if r.status_code == 200:

    total = int(r.headers.get("content-length", 0))
    downloaded = 0

    with open(ANNOTATION_ZIP, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if total:
                    percent = downloaded * 100 / total
                    print(
                        f"\rDownloaded: {percent:.1f}%",
                        end=""
                    )

    print("\n\nDownload complete.")
    print(
        "Size:",
        round(os.path.getsize(ANNOTATION_ZIP) / (1024**2), 2),
        "MB"
    )

else:
    print("Download failed:", r.status_code)

Destination: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ami_public_manual_1.6.2.zip
Status: 200
Downloaded: 100.0%

Download complete.
Size: 21.83 MB


In [25]:
import zipfile
import os

print("Checking annotation archive...")

with zipfile.ZipFile(ANNOTATION_ZIP, "r") as z:
    files = z.namelist()

print("Total files:", len(files))

print("\nFirst 30 files:")
for f in files[:30]:
    print(f)

Checking annotation archive...
Total files: 5183

First 30 files:
00README_MANUAL.txt
abstractive/
abstractive/ES2008a.abssumm.xml
abstractive/ES2006a.abssumm.xml
abstractive/TS3007c.abssumm.xml
abstractive/ES2005b.abssumm.xml
abstractive/ES2005c.abssumm.xml
abstractive/TS3004d.abssumm.xml
abstractive/IS1006d.abssumm.xml
abstractive/IB4003.abssumm.xml
abstractive/TS3011a.abssumm.xml
abstractive/ES2010a.abssumm.xml
abstractive/ES2011b.abssumm.xml
abstractive/ES2009d.abssumm.xml
abstractive/TS3005c.abssumm.xml
abstractive/ES2002c.abssumm.xml
abstractive/TS3003d.abssumm.xml
abstractive/ES2011a.abssumm.xml
abstractive/ES2009b.abssumm.xml
abstractive/ES2008d.abssumm.xml
abstractive/IS1004b.abssumm.xml
abstractive/TS3007a.abssumm.xml
abstractive/IS1006c.abssumm.xml
abstractive/ES2012d.abssumm.xml
abstractive/ES2015b.abssumm.xml
abstractive/TS3006d.abssumm.xml
abstractive/ES2005a.abssumm.xml
abstractive/IS1008a.abssumm.xml
abstractive/IS1008c.abssumm.xml
abstractive/IS1005a.abssumm.xml


In [26]:
es2004a_files = [
    f for f in files
    if "ES2004a" in f
]

print("ES2004a annotation files:", len(es2004a_files))

for f in es2004a_files:
    print(f)

ES2004a annotation files: 35
abstractive/ES2004a.abssumm.xml
argumentation/ae/ES2004a.D.argumentstructs.xml
argumentation/ae/ES2004a.B.argumentstructs.xml
argumentation/ae/ES2004a.A.argumentstructs.xml
argumentation/ae/ES2004a.C.argumentstructs.xml
argumentation/ar/ES2004a.argumentationrels.xml
argumentation/dis/ES2004a.discussions.xml
dialogueActs/ES2004a.C.dialog-act.xml
dialogueActs/ES2004a.adjacency-pairs.xml
dialogueActs/ES2004a.A.dialog-act.xml
dialogueActs/ES2004a.D.dialog-act.xml
dialogueActs/ES2004a.B.dialog-act.xml
extractive/ES2004a.extsumm.xml
extractive/ES2004a.summlink.xml
movement/ES2004a.D.movement.xml
movement/ES2004a.B.movement.xml
movement/ES2004a.C.movement.xml
movement/ES2004a.A.movement.xml
namedEntities/ES2004a.B.ne.xml
namedEntities/ES2004a.D.ne.xml
namedEntities/ES2004a.C.ne.xml
namedEntities/ES2004a.A.ne.xml
participantSummaries/ES2004a.B.summ.xml
participantSummaries/ES2004a.D.summ.xml
participantSummaries/ES2004a.A.summ.xml
participantSummaries/ES2004a.C.sum

In [27]:
import zipfile
import os

ES_DIR = os.path.join(AMI_DIR, "ES2004a")

annotation_files_to_extract = [
    "abstractive/ES2004a.abssumm.xml",
    "extractive/ES2004a.extsumm.xml",
    "extractive/ES2004a.summlink.xml",
    "segments/ES2004a.A.segments.xml",
    "segments/ES2004a.B.segments.xml",
    "segments/ES2004a.C.segments.xml",
    "segments/ES2004a.D.segments.xml",
    "words/ES2004a.A.words.xml",
    "words/ES2004a.B.words.xml",
    "words/ES2004a.C.words.xml",
    "words/ES2004a.D.words.xml",
    "dialogueActs/ES2004a.A.dialog-act.xml",
    "dialogueActs/ES2004a.B.dialog-act.xml",
    "dialogueActs/ES2004a.C.dialog-act.xml",
    "dialogueActs/ES2004a.D.dialog-act.xml",
    "participantSummaries/ES2004a.A.summ.xml",
    "participantSummaries/ES2004a.B.summ.xml",
    "participantSummaries/ES2004a.C.summ.xml",
    "participantSummaries/ES2004a.D.summ.xml",
]

with zipfile.ZipFile(ANNOTATION_ZIP, "r") as z:

    for file_name in annotation_files_to_extract:

        z.extract(
            file_name,
            ES_DIR
        )

print("Extraction complete.")

for root, dirs, files_in_dir in os.walk(ES_DIR):
    for f in files_in_dir:
        if f.endswith(".xml"):
            print(os.path.join(root, f))

Extraction complete.
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/Jslides.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/cardSlides.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/abstractive/ES2004a.abssumm.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/extractive/ES2004a.extsumm.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/extractive/ES2004a.summlink.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/segments/ES2004a.A.segments.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/segments/ES2004a.B.segments.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/segments/ES2004a.C.segments.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/segments/ES2004a.D.segments.xml
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/words/ES2004a.A.words.xml

In [28]:
import xml.etree.ElementTree as ET
import os
import re

WORDS_DIR = os.path.join(
    AMI_DIR,
    "ES2004a",
    "words"
)

word_files = sorted([
    os.path.join(WORDS_DIR, f)
    for f in os.listdir(WORDS_DIR)
    if f.endswith(".xml")
])

print("Word annotation files:", len(word_files))

for f in word_files:
    print(os.path.basename(f))

Word annotation files: 4
ES2004a.A.words.xml
ES2004a.B.words.xml
ES2004a.C.words.xml
ES2004a.D.words.xml


In [29]:
import xml.etree.ElementTree as ET
import os
import re

TRANSCRIPT_DIR = os.path.join(
    AMI_DIR,
    "ES2004a",
    "transcript"
)

os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

records = []

for xml_file in word_files:
    speaker = os.path.basename(xml_file).split(".")[1]

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for elem in root.iter():
        # AMI word annotations contain word elements
        if elem.tag.lower().endswith("w") or elem.tag.lower() == "word":
            text = elem.text.strip() if elem.text else ""

            if not text:
                continue

            start = elem.attrib.get("starttime")
            end = elem.attrib.get("endtime")

            # Some AMI XML versions use href/id references instead,
            # so keep only records where timing is directly available.
            if start is not None and end is not None:
                records.append({
                    "speaker": speaker,
                    "start": float(start),
                    "end": float(end),
                    "text": text
                })

print("Timestamped words found:", len(records))

records = sorted(
    records,
    key=lambda x: (x["start"], x["speaker"])
)

print("\nFirst 20 records:\n")

for r in records[:20]:
    print(
        f"[{r['start']:.2f} - {r['end']:.2f}] "
        f"{r['speaker']}: {r['text']}"
    )

Timestamped words found: 3135

First 20 records:

[0.37 - 0.95] A: Hmm
[0.95 - 1.53] A: hmm
[1.53 - 1.76] A: hmm
[1.76 - 1.76] A: .
[10.99 - 11.02] B: Are
[11.02 - 12.13] B: we
[12.13 - 12.29] B: we're
[12.29 - 12.42] B: not
[12.42 - 12.62] B: allowed
[12.62 - 12.70] B: to
[12.70 - 12.84] B: dim
[12.84 - 12.91] B: the
[12.91 - 13.18] B: lights
[13.18 - 13.31] B: so
[13.31 - 13.53] B: people
[13.53 - 13.71] B: can
[13.71 - 13.81] B: see
[13.81 - 13.96] B: that
[13.96 - 13.99] B: a
[13.99 - 14.15] B: bit


In [30]:
import json

transcript_json_path = os.path.join(
    TRANSCRIPT_DIR,
    "ES2004a_timestamped_transcript.json"
)

with open(transcript_json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print("Saved:")
print(transcript_json_path)
print("Records:", len(records))

Saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/transcript/ES2004a_timestamped_transcript.json
Records: 3135


In [31]:
transcript_txt_path = os.path.join(
    TRANSCRIPT_DIR,
    "ES2004a_transcript.txt"
)

with open(transcript_txt_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(
            f"[{r['start']:.2f} - {r['end']:.2f}] "
            f"{r['speaker']}: {r['text']}\n"
        )

print("Saved:")
print(transcript_txt_path)

Saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/transcript/ES2004a_transcript.txt


In [32]:
SUMMARY_PATH = os.path.join(
    AMI_DIR,
    "ES2004a",
    "abstractive",
    "ES2004a.abssumm.xml"
)

print("Summary file:")
print(SUMMARY_PATH)

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary_xml = f.read()

print("\nFirst 4000 characters:\n")
print(summary_xml[:4000])

Summary file:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/abstractive/ES2004a.abssumm.xml

First 4000 characters:

<?xml version="1.0" encoding="ISO-8859-1" standalone="yes"?>
<nite:root xmlns:nite="http://nite.sourceforge.net/">
<abstract nite:id="ES2004a.JacquelinePalmer.abstract.1">

<sentence nite:id="ES2004a.JacquelinePalmer.s.1">The Project Manager gave an introduction to the goal of the project, to create a trendy yet user-friendly remote.</sentence> <sentence nite:id="ES2004a.JacquelinePalmer.s.2">She presented a long-range agenda for the whole project.</sentence> <sentence nite:id="ES2004a.JacquelinePalmer.s.3">The group introduced themselves to each other and practiced with the meeting room tools by drawing on the board.</sentence> <sentence nite:id="ES2004a.JacquelinePalmer.s.4">The Project Manager presented the project budget, the projected price point, and the projected profit aim for the project.</sentence> <sentence nite:id="ES2004a.JacquelinePal

In [33]:
import xml.etree.ElementTree as ET
import os

tree = ET.parse(SUMMARY_PATH)
root = tree.getroot()

summary_sections = {
    "ABSTRACT": [],
    "ACTIONS": [],
    "DECISIONS": [],
    "PROBLEMS": []
}

for section in root:
    section_name = section.tag.upper()

    if section_name in summary_sections:
        for sentence in section.findall("sentence"):
            text = "".join(sentence.itertext()).strip()

            if text:
                summary_sections[section_name].append(text)

REFERENCE_SUMMARY_PATH = os.path.join(
    AMI_DIR,
    "ES2004a",
    "ES2004a_reference_summary.txt"
)

with open(REFERENCE_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for section, sentences in summary_sections.items():
        f.write(f"\n## {section}\n\n")

        for sentence in sentences:
            f.write(sentence + "\n")

print("Reference summary saved:")
print(REFERENCE_SUMMARY_PATH)

print("\nSections:")
for section, sentences in summary_sections.items():
    print(f"{section}: {len(sentences)} sentences")

Reference summary saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_reference_summary.txt

Sections:
ABSTRACT: 7 sentences
ACTIONS: 1 sentences
DECISIONS: 1 sentences
PROBLEMS: 1 sentences


In [34]:
with open(REFERENCE_SUMMARY_PATH, "r", encoding="utf-8") as f:
    print(f.read())


## ABSTRACT

The Project Manager gave an introduction to the goal of the project, to create a trendy yet user-friendly remote.
She presented a long-range agenda for the whole project.
The group introduced themselves to each other and practiced with the meeting room tools by drawing on the board.
The Project Manager presented the project budget, the projected price point, and the projected profit aim for the project.
Then the group began a discussion about their own experiences with remote controls to generate initial design ideas for making the product user-friendly.
They discussed grouping features into a menu and adding an LCD display.
They also discussed the look of various materials that may be used in the design, in keeping with the company's goal to create fashionable electronics.

## ACTIONS

The group will prepare for the functional design meeting to discuss the components and functions of the product.

## DECISIONS

NA.

## PROBLEMS

There may not be enough money in the budge

In [35]:
import json
from datetime import datetime

verification = {
    "meeting_id": "ES2004a",
    "video": True,
    "audio": True,
    "slides": True,
    "timestamped_transcript": True,
    "reference_summary": True,
    "dialogue_act_annotations": True,
    "abstract_sentences": len(summary_sections["ABSTRACT"]),
    "action_sentences": len(summary_sections["ACTIONS"]),
    "decision_sentences": len(summary_sections["DECISIONS"]),
    "problem_sentences": len(summary_sections["PROBLEMS"]),
    "status": "READY_FOR_PIPELINE",
    "verified_on": datetime.now().isoformat()
}

verification_path = os.path.join(
    AMI_DIR,
    "ES2004a",
    "ES2004a_verification.json"
)

with open(verification_path, "w", encoding="utf-8") as f:
    json.dump(verification, f, indent=2)

print(json.dumps(verification, indent=2))
print("\nSaved:", verification_path)

{
  "meeting_id": "ES2004a",
  "video": true,
  "audio": true,
  "slides": true,
  "timestamped_transcript": true,
  "reference_summary": true,
  "dialogue_act_annotations": true,
  "abstract_sentences": 7,
  "action_sentences": 1,
  "decision_sentences": 1,
  "problem_sentences": 1,
  "status": "READY_FOR_PIPELINE",
  "verified_on": "2026-08-26T07:55:16.036632"
}

Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_verification.json


Perfect. ES2004a is officially our first verified meeting. ✅

Step 27 — Now prepare the remaining meetings automatically

We will first test the automation on ES2004b only.

In [36]:
import os
import requests
import re
import html

def get_ami_download_script(meeting_id):
    """
    Ask the official AMI download server to generate
    the download script for one meeting.
    """

    session = meeting_id[:-1]   # ES2004
    participant = meeting_id[-1]  # b

    payload = [
        ("sessionID", session),
        ("ES", participant),
        ("mediaType", "DivXvideos"),
        ("mediaType", "Mix-Headset"),
        ("mediaType", "slides"),
        ("mediaType", "smi"),
    ]

    r = requests.post(
        "https://homepages.inf.ed.ac.uk/cgi/simonk/AMI/estimate.cgi",
        data=payload,
        timeout=60
    )

    r.raise_for_status()

    match = re.search(
        r"fillText\('wgetsh','(.*?)'\)",
        r.text,
        re.DOTALL
    )

    if not match:
        raise RuntimeError(
            f"Could not obtain AMI download script for {meeting_id}"
        )

    script = match.group(1)
    script = script.replace("\\n", "\n")
    script = html.unescape(script)

    return script


test_script = get_ami_download_script("ES2004b")

print("Download script obtained successfully.")
print("Script length:", len(test_script))
print("\nFirst 1000 characters:\n")
print(test_script[:1000])

Download script obtained successfully.
Script length: 24360

First 1000 characters:

#!/bin/sh
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.PreferredOverview.avi
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Closeup1.avi
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Closeup2.avi
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Closeup3.avi
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Closeup4.avi
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Corner.avi
wget    -P amicorpus/ES2004b/video https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Overhead

In [37]:
import re
from urllib.parse import urlparse

meeting_id = "ES2004b"

video_urls = []
audio_urls = []
slide_urls = []

for line in test_script.splitlines():
    if not line.startswith("wget"):
        continue

    parts = line.split()
    url = parts[-1]

    if f"/{meeting_id}/video/" in url:
        video_urls.append(url)

    elif f"/{meeting_id}/audio/" in url:
        audio_urls.append(url)

    elif f"/{meeting_id}/slides/" in url:
        slide_urls.append(url)

print("Video files:", len(video_urls))
print("Audio files:", len(audio_urls))
print("Slide files:", len(slide_urls))

print("\nVideo files:")
for url in video_urls:
    print(os.path.basename(urlparse(url).path))

print("\nAudio files:")
for url in audio_urls:
    print(os.path.basename(urlparse(url).path))

Video files: 7
Audio files: 1
Slide files: 167

Video files:
ES2004b.PreferredOverview.avi
ES2004b.Closeup1.avi
ES2004b.Closeup2.avi
ES2004b.Closeup3.avi
ES2004b.Closeup4.avi
ES2004b.Corner.avi
ES2004b.Overhead.avi

Audio files:
ES2004b.Mix-Headset.wav


In [38]:
import requests
import os
from urllib.parse import urlparse

video_dir = os.path.join(
    AMI_DIR,
    "ES2004b",
    "video"
)

os.makedirs(video_dir, exist_ok=True)

accessible_video = None

# Try PreferredOverview first, then the other camera streams
for url in video_urls:

    filename = os.path.basename(urlparse(url).path)

    print("Testing:", filename)

    try:
        r = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            stream=True,
            timeout=60
        )

        print("  Status:", r.status_code)

        if r.status_code == 200:
            accessible_video = url
            print("  ✓ Accessible")
            r.close()
            break
        else:
            print("  ✗ Not accessible")

        r.close()

    except Exception as e:
        print("  ✗ Error:", str(e))

print("\nSelected video:")
print(accessible_video)

Testing: ES2004b.PreferredOverview.avi
  Status: 403
  ✗ Not accessible
Testing: ES2004b.Closeup1.avi
  Status: 200
  ✓ Accessible

Selected video:
https://groups.inf.ed.ac.uk/ami/AMICorpusMirror//amicorpus/ES2004b/video/ES2004b.Closeup1.avi


In [39]:
import os
import requests
from urllib.parse import urlparse

MEETING_ID = "ES2004b"

MEETING_DIR = os.path.join(AMI_DIR, MEETING_ID)

VIDEO_DIR = os.path.join(MEETING_DIR, "video")
AUDIO_DIR = os.path.join(MEETING_DIR, "audio")
SLIDES_DIR = os.path.join(MEETING_DIR, "slides")

for d in [VIDEO_DIR, AUDIO_DIR, SLIDES_DIR]:
    os.makedirs(d, exist_ok=True)


def download_file(url, output_dir):
    filename = os.path.basename(urlparse(url).path)
    output_path = os.path.join(output_dir, filename)

    if os.path.exists(output_path):
        print("Already exists:", filename)
        return True

    print("\nDownloading:", filename)

    r = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        stream=True,
        timeout=120
    )

    if r.status_code != 200:
        print("FAILED:", r.status_code)
        r.close()
        return False

    with open(output_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    r.close()

    size_mb = os.path.getsize(output_path) / (1024 ** 2)

    print(f"✓ {filename} — {size_mb:.2f} MB")

    return True


# 1. Download the selected video
download_file(accessible_video, VIDEO_DIR)


# 2. Download headset-mix audio
download_file(audio_urls[0], AUDIO_DIR)


# 3. Download slides
successful_slides = 0

for url in slide_urls:
    if download_file(url, SLIDES_DIR):
        successful_slides += 1

print("\n==============================")
print("ES2004b download complete")
print("==============================")
print("Slides downloaded:", successful_slides)


Downloading: ES2004b.Closeup1.avi
✓ ES2004b.Closeup1.avi — 52.55 MB

Downloading: ES2004b.Mix-Headset.wav
✓ ES2004b.Mix-Headset.wav — 71.58 MB

Downloading: ES2004b.1002.84__1023.47.txt
✓ ES2004b.1002.84__1023.47.txt — 0.00 MB

Downloading: ES2004b.1023.47__1059.35.txt
✓ ES2004b.1023.47__1059.35.txt — 0.00 MB

Downloading: ES2004b.1059.35__1067.67.txt
✓ ES2004b.1059.35__1067.67.txt — 0.00 MB

Downloading: ES2004b.1067.67__1083.49.txt
✓ ES2004b.1067.67__1083.49.txt — 0.00 MB

Downloading: ES2004b.1083.49__1117.12.txt
✓ ES2004b.1083.49__1117.12.txt — 0.00 MB

Downloading: ES2004b.1117.12__1122.97.txt
✓ ES2004b.1117.12__1122.97.txt — 0.00 MB

Downloading: ES2004b.1122.97__1133.83.txt
✓ ES2004b.1122.97__1133.83.txt — 0.00 MB

Downloading: ES2004b.113.08__118.82.txt
✓ ES2004b.113.08__118.82.txt — 0.00 MB

Downloading: ES2004b.113.08__160.88.jpg
✓ ES2004b.113.08__160.88.jpg — 0.16 MB

Downloading: ES2004b.1133.83__1146.23.txt
✓ ES2004b.1133.83__1146.23.txt — 0.00 MB

Downloading: ES2004b.11

In [40]:
for resource in ["video", "audio", "slides"]:

    path = os.path.join(
        AMI_DIR,
        "ES2004b",
        resource
    )

    files_here = os.listdir(path)

    total_size = sum(
        os.path.getsize(os.path.join(path, f))
        for f in files_here
    )

    print(
        f"{resource.upper():8s} | "
        f"Files: {len(files_here):3d} | "
        f"Size: {total_size / (1024**2):.2f} MB"
    )

VIDEO    | Files:   1 | Size: 52.55 MB
AUDIO    | Files:   1 | Size: 71.58 MB
SLIDES   | Files: 167 | Size: 4.84 MB


In [41]:
MEETING_ID = "ES2004b"

annotation_files = [
    f for f in files
    if MEETING_ID in f
]

print("Annotation files found:", len(annotation_files))

for f in annotation_files:
    print(f)

Annotation files found: 35
abstractive/ES2004b.abssumm.xml
argumentation/ae/ES2004b.B.argumentstructs.xml
argumentation/ae/ES2004b.D.argumentstructs.xml
argumentation/ae/ES2004b.A.argumentstructs.xml
argumentation/ae/ES2004b.C.argumentstructs.xml
argumentation/ar/ES2004b.argumentationrels.xml
argumentation/dis/ES2004b.discussions.xml
dialogueActs/ES2004b.C.dialog-act.xml
dialogueActs/ES2004b.A.dialog-act.xml
dialogueActs/ES2004b.adjacency-pairs.xml
dialogueActs/ES2004b.D.dialog-act.xml
dialogueActs/ES2004b.B.dialog-act.xml
extractive/ES2004b.extsumm.xml
extractive/ES2004b.summlink.xml
movement/ES2004b.D.movement.xml
movement/ES2004b.C.movement.xml
movement/ES2004b.B.movement.xml
movement/ES2004b.A.movement.xml
namedEntities/ES2004b.C.ne.xml
namedEntities/ES2004b.D.ne.xml
namedEntities/ES2004b.A.ne.xml
namedEntities/ES2004b.B.ne.xml
participantSummaries/ES2004b.B.summ.xml
participantSummaries/ES2004b.C.summ.xml
participantSummaries/ES2004b.A.summ.xml
participantSummaries/ES2004b.D.summ.

In [42]:
MEETING_ID = "ES2004b"

annotation_files_to_extract = [
    f"abstractive/{MEETING_ID}.abssumm.xml",
    f"extractive/{MEETING_ID}.extsumm.xml",
    f"extractive/{MEETING_ID}.summlink.xml",

    f"segments/{MEETING_ID}.A.segments.xml",
    f"segments/{MEETING_ID}.B.segments.xml",
    f"segments/{MEETING_ID}.C.segments.xml",
    f"segments/{MEETING_ID}.D.segments.xml",

    f"words/{MEETING_ID}.A.words.xml",
    f"words/{MEETING_ID}.B.words.xml",
    f"words/{MEETING_ID}.C.words.xml",
    f"words/{MEETING_ID}.D.words.xml",

    f"dialogueActs/{MEETING_ID}.A.dialog-act.xml",
    f"dialogueActs/{MEETING_ID}.B.dialog-act.xml",
    f"dialogueActs/{MEETING_ID}.C.dialog-act.xml",
    f"dialogueActs/{MEETING_ID}.D.dialog-act.xml",

    f"participantSummaries/{MEETING_ID}.A.summ.xml",
    f"participantSummaries/{MEETING_ID}.B.summ.xml",
    f"participantSummaries/{MEETING_ID}.C.summ.xml",
    f"participantSummaries/{MEETING_ID}.D.summ.xml",
]

MEETING_DIR = os.path.join(AMI_DIR, MEETING_ID)

with zipfile.ZipFile(ANNOTATION_ZIP, "r") as z:

    available = set(z.namelist())

    for file_name in annotation_files_to_extract:

        if file_name in available:
            z.extract(file_name, MEETING_DIR)
        else:
            print("MISSING:", file_name)

print("Extraction complete.")

Extraction complete.


In [43]:
WORDS_DIR = os.path.join(
    AMI_DIR,
    MEETING_ID,
    "words"
)

TRANSCRIPT_DIR = os.path.join(
    AMI_DIR,
    MEETING_ID,
    "transcript"
)

os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

word_files = sorted([
    os.path.join(WORDS_DIR, f)
    for f in os.listdir(WORDS_DIR)
    if f.endswith(".xml")
])

records = []

for xml_file in word_files:

    speaker = os.path.basename(xml_file).split(".")[1]

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for elem in root.iter():

        if elem.tag.lower().endswith("w") or elem.tag.lower() == "word":

            text = elem.text.strip() if elem.text else ""

            if not text:
                continue

            start = elem.attrib.get("starttime")
            end = elem.attrib.get("endtime")

            if start is not None and end is not None:
                records.append({
                    "speaker": speaker,
                    "start": float(start),
                    "end": float(end),
                    "text": text
                })

records = sorted(
    records,
    key=lambda x: (x["start"], x["speaker"])
)

print("Timestamped words:", len(records))

for r in records[:15]:
    print(
        f"[{r['start']:.2f} - {r['end']:.2f}] "
        f"{r['speaker']}: {r['text']}"
    )

Timestamped words: 7681
[0.20 - 4.83] B: Help
[4.83 - 4.83] B: .
[8.18 - 8.27] B: It's
[8.27 - 8.44] B: up
[8.44 - 9.82] B: there
[9.82 - 9.82] B: ?
[9.82 - 9.98] B: That
[9.98 - 10.33] B: screen's
[10.33 - 10.80] B: black
[10.80 - 10.80] B: .
[37.05 - 37.56] B: Alright
[37.56 - 37.56] B: ,
[37.56 - 38.83] B: okay
[38.83 - 38.83] B: .
[38.83 - 39.28] B: Okay


In [44]:
import json

transcript_json_path = os.path.join(
    TRANSCRIPT_DIR,
    "ES2004b_timestamped_transcript.json"
)

with open(transcript_json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

transcript_txt_path = os.path.join(
    TRANSCRIPT_DIR,
    "ES2004b_transcript.txt"
)

with open(transcript_txt_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(
            f"[{r['start']:.2f} - {r['end']:.2f}] "
            f"{r['speaker']}: {r['text']}\n"
        )

print("JSON saved:", transcript_json_path)
print("TXT saved:", transcript_txt_path)

JSON saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/transcript/ES2004b_timestamped_transcript.json
TXT saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/transcript/ES2004b_transcript.txt


In [45]:
SUMMARY_PATH = os.path.join(
    AMI_DIR,
    MEETING_ID,
    "abstractive",
    f"{MEETING_ID}.abssumm.xml"
)

tree = ET.parse(SUMMARY_PATH)
root = tree.getroot()

summary_sections = {
    "ABSTRACT": [],
    "ACTIONS": [],
    "DECISIONS": [],
    "PROBLEMS": []
}

for section in root:
    section_name = section.tag.upper()

    if section_name in summary_sections:
        for sentence in section.findall("sentence"):
            text = "".join(sentence.itertext()).strip()

            if text:
                summary_sections[section_name].append(text)

REFERENCE_SUMMARY_PATH = os.path.join(
    AMI_DIR,
    MEETING_ID,
    f"{MEETING_ID}_reference_summary.txt"
)

with open(REFERENCE_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for section, sentences in summary_sections.items():
        f.write(f"\n## {section}\n\n")

        for sentence in sentences:
            f.write(sentence + "\n")

print("Reference summary saved:")
print(REFERENCE_SUMMARY_PATH)

print("\nSummary statistics:")
for section, sentences in summary_sections.items():
    print(f"{section}: {len(sentences)} sentences")

Reference summary saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/ES2004b_reference_summary.txt

Summary statistics:
ABSTRACT: 9 sentences
ACTIONS: 1 sentences
DECISIONS: 4 sentences
PROBLEMS: 1 sentences


In [46]:
with open(REFERENCE_SUMMARY_PATH, "r", encoding="utf-8") as f:
    print(f.read())


## ABSTRACT

The Industrial Designer gave his presentation on the basic functions of the remote.
He presented the basic components that remotes share and suggested that smaller batteries be considered in the product design.
The User Interface Designer presented his ideas for making the remote easy-to-use; he discussed using a simple design and hiding complicated features from the main interface.
The Marketing Expert presented the findings from a lab study on user requirements for a remote control device, and discussed users' demand for a simple interface and advanced technology.
The Project Manager presented the new requirements that the remote not include a teletext function, that it be used only to control television, and that it include the company image in its design.
The group narrowed down their target marketing group to the youth market.
They discussed the functions the remote will have, including Video Plus capability and rechargeable batteries.
A customer service plan was sug

In [48]:
import json
from datetime import datetime

verification = {
    "meeting_id": MEETING_ID,
    "video": True,
    "audio": True,
    "slides": True,
    "timestamped_transcript": True,
    "reference_summary": True,
    "dialogue_act_annotations": True,
    "timestamped_words": len(records),
    "abstract_sentences": len(summary_sections["ABSTRACT"]),
    "action_sentences": len(summary_sections["ACTIONS"]),
    "decision_sentences": len(summary_sections["DECISIONS"]),
    "problem_sentences": len(summary_sections["PROBLEMS"]),
    "status": "READY_FOR_PIPELINE",
    "verified_on": datetime.now().isoformat()
}

verification_path = os.path.join(
    AMI_DIR,
    MEETING_ID,
    f"{MEETING_ID}_verification.json"
)

with open(verification_path, "w", encoding="utf-8") as f:
    json.dump(verification, f, indent=2)

print(json.dumps(verification, indent=2))
print("\nSaved:", verification_path)

{
  "meeting_id": "ES2004b",
  "video": true,
  "audio": true,
  "slides": true,
  "timestamped_transcript": true,
  "reference_summary": true,
  "dialogue_act_annotations": true,
  "timestamped_words": 7681,
  "abstract_sentences": 9,
  "action_sentences": 1,
  "decision_sentences": 4,
  "problem_sentences": 1,
  "status": "READY_FOR_PIPELINE",
  "verified_on": "2026-08-26T08:07:15.958577"
}

Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/ES2004b_verification.json


In [49]:
import os
import re
import json
import html
import zipfile
import requests
import xml.etree.ElementTree as ET
from urllib.parse import urlparse
from datetime import datetime


AMI_BASE = "https://homepages.inf.ed.ac.uk/cgi/simonk/AMI/estimate.cgi"


def get_ami_download_script(meeting_id):

    session = meeting_id[:-1]
    participant = meeting_id[-1]

    payload = [
        ("sessionID", session),
        ("ES", participant),
        ("mediaType", "DivXvideos"),
        ("mediaType", "Mix-Headset"),
        ("mediaType", "slides"),
        ("mediaType", "smi"),
    ]

    response = requests.post(
        AMI_BASE,
        data=payload,
        timeout=60
    )

    response.raise_for_status()

    match = re.search(
        r"fillText\('wgetsh','(.*?)'\)",
        response.text,
        re.DOTALL
    )

    if not match:
        raise RuntimeError(
            f"AMI download script not found for {meeting_id}"
        )

    script = match.group(1)
    script = script.replace("\\n", "\n")
    script = html.unescape(script)

    return script


def extract_urls(script, meeting_id):

    video_urls = []
    audio_urls = []
    slide_urls = []

    for line in script.splitlines():

        if not line.startswith("wget"):
            continue

        parts = line.split()

        if not parts:
            continue

        url = parts[-1]

        if f"/{meeting_id}/video/" in url:
            video_urls.append(url)

        elif f"/{meeting_id}/audio/" in url:
            audio_urls.append(url)

        elif f"/{meeting_id}/slides/" in url:
            slide_urls.append(url)

    return video_urls, audio_urls, slide_urls


def find_accessible_video(video_urls):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    for url in video_urls:

        filename = os.path.basename(
            urlparse(url).path
        )

        print("Testing video:", filename)

        try:

            response = requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=60
            )

            status = response.status_code
            response.close()

            if status == 200:

                print("  ✓ Accessible")

                return url

            print("  ✗", status)

        except Exception as e:

            print("  ✗ Error:", e)

    return None


def download_file(url, output_dir):

    os.makedirs(output_dir, exist_ok=True)

    filename = os.path.basename(
        urlparse(url).path
    )

    output_path = os.path.join(
        output_dir,
        filename
    )

    if os.path.exists(output_path):

        print("Already exists:", filename)

        return output_path

    print("Downloading:", filename)

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        stream=True,
        timeout=120
    )

    if response.status_code != 200:

        response.close()

        print(
            "FAILED:",
            response.status_code,
            filename
        )

        return None

    with open(output_path, "wb") as f:

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):

            if chunk:
                f.write(chunk)

    response.close()

    size_mb = os.path.getsize(
        output_path
    ) / (1024 ** 2)

    print(
        f"  ✓ {size_mb:.2f} MB"
    )

    return output_path


def extract_annotations(
    meeting_id,
    meeting_dir,
    annotation_zip
):

    required = [
        f"abstractive/{meeting_id}.abssumm.xml",

        f"extractive/{meeting_id}.extsumm.xml",
        f"extractive/{meeting_id}.summlink.xml",

        f"segments/{meeting_id}.A.segments.xml",
        f"segments/{meeting_id}.B.segments.xml",
        f"segments/{meeting_id}.C.segments.xml",
        f"segments/{meeting_id}.D.segments.xml",

        f"words/{meeting_id}.A.words.xml",
        f"words/{meeting_id}.B.words.xml",
        f"words/{meeting_id}.C.words.xml",
        f"words/{meeting_id}.D.words.xml",

        f"dialogueActs/{meeting_id}.A.dialog-act.xml",
        f"dialogueActs/{meeting_id}.B.dialog-act.xml",
        f"dialogueActs/{meeting_id}.C.dialog-act.xml",
        f"dialogueActs/{meeting_id}.D.dialog-act.xml",

        f"participantSummaries/{meeting_id}.A.summ.xml",
        f"participantSummaries/{meeting_id}.B.summ.xml",
        f"participantSummaries/{meeting_id}.C.summ.xml",
        f"participantSummaries/{meeting_id}.D.summ.xml",
    ]

    with zipfile.ZipFile(
        annotation_zip,
        "r"
    ) as z:

        available = set(z.namelist())

        extracted = 0

        for file_name in required:

            if file_name in available:

                z.extract(
                    file_name,
                    meeting_dir
                )

                extracted += 1

            else:

                print(
                    "MISSING:",
                    file_name
                )

    return extracted


def create_timestamped_transcript(
    meeting_id,
    meeting_dir
):

    words_dir = os.path.join(
        meeting_dir,
        "words"
    )

    transcript_dir = os.path.join(
        meeting_dir,
        "transcript"
    )

    os.makedirs(
        transcript_dir,
        exist_ok=True
    )

    word_files = sorted([
        os.path.join(words_dir, f)
        for f in os.listdir(words_dir)
        if f.endswith(".xml")
    ])

    records = []

    for xml_file in word_files:

        speaker = os.path.basename(
            xml_file
        ).split(".")[1]

        tree = ET.parse(xml_file)
        root = tree.getroot()

        for elem in root.iter():

            if (
                elem.tag.lower().endswith("w")
                or elem.tag.lower() == "word"
            ):

                text = (
                    elem.text.strip()
                    if elem.text
                    else ""
                )

                if not text:
                    continue

                start = elem.attrib.get(
                    "starttime"
                )

                end = elem.attrib.get(
                    "endtime"
                )

                if (
                    start is not None
                    and end is not None
                ):

                    records.append({
                        "speaker": speaker,
                        "start": float(start),
                        "end": float(end),
                        "text": text
                    })

    records.sort(
        key=lambda x: (
            x["start"],
            x["speaker"]
        )
    )

    json_path = os.path.join(
        transcript_dir,
        f"{meeting_id}_timestamped_transcript.json"
    )

    txt_path = os.path.join(
        transcript_dir,
        f"{meeting_id}_transcript.txt"
    )

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    with open(
        txt_path,
        "w",
        encoding="utf-8"
    ) as f:

        for r in records:

            f.write(
                f"[{r['start']:.2f} - "
                f"{r['end']:.2f}] "
                f"{r['speaker']}: "
                f"{r['text']}\n"
            )

    return records


def create_reference_summary(
    meeting_id,
    meeting_dir
):

    summary_path = os.path.join(
        meeting_dir,
        "abstractive",
        f"{meeting_id}.abssumm.xml"
    )

    tree = ET.parse(summary_path)
    root = tree.getroot()

    sections = {
        "ABSTRACT": [],
        "ACTIONS": [],
        "DECISIONS": [],
        "PROBLEMS": []
    }

    for section in root:

        name = section.tag.upper()

        if name in sections:

            for sentence in section.findall(
                "sentence"
            ):

                text = "".join(
                    sentence.itertext()
                ).strip()

                if text:
                    sections[name].append(
                        text
                    )

    output_path = os.path.join(
        meeting_dir,
        f"{meeting_id}_reference_summary.txt"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for section, sentences in sections.items():

            f.write(
                f"\n## {section}\n\n"
            )

            for sentence in sentences:

                f.write(
                    sentence + "\n"
                )

    return sections


print("Reusable AMI preparation pipeline defined successfully.")

Reusable AMI preparation pipeline defined successfully.


In [50]:
script = get_ami_download_script("ES2004c")

videos, audios, slides = extract_urls(
    script,
    "ES2004c"
)

print("ES2004c")
print("Videos:", len(videos))
print("Audio:", len(audios))
print("Slides:", len(slides))

ES2004c
Videos: 7
Audio: 1
Slides: 127


In [51]:
def prepare_ami_meeting(meeting_id):

    print("=" * 60)
    print(f"Preparing {meeting_id}")
    print("=" * 60)

    meeting_dir = os.path.join(
        AMI_DIR,
        meeting_id
    )

    video_dir = os.path.join(
        meeting_dir,
        "video"
    )

    audio_dir = os.path.join(
        meeting_dir,
        "audio"
    )

    slides_dir = os.path.join(
        meeting_dir,
        "slides"
    )

    os.makedirs(video_dir, exist_ok=True)
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(slides_dir, exist_ok=True)

    # --------------------------------
    # 1. Get AMI download script
    # --------------------------------

    print("\n[1/7] Getting AMI download script...")

    script = get_ami_download_script(
        meeting_id
    )

    videos, audios, slides = extract_urls(
        script,
        meeting_id
    )

    print("Videos:", len(videos))
    print("Audio:", len(audios))
    print("Slides:", len(slides))

    # --------------------------------
    # 2. Select accessible video
    # --------------------------------

    print("\n[2/7] Selecting accessible video...")

    selected_video = find_accessible_video(
        videos
    )

    if selected_video is None:
        raise RuntimeError(
            f"No accessible video found for {meeting_id}"
        )

    print(
        "Selected:",
        os.path.basename(
            urlparse(selected_video).path
        )
    )

    # --------------------------------
    # 3. Download video
    # --------------------------------

    print("\n[3/7] Downloading video...")

    download_file(
        selected_video,
        video_dir
    )

    # --------------------------------
    # 4. Download audio
    # --------------------------------

    print("\n[4/7] Downloading audio...")

    if audios:
        download_file(
            audios[0],
            audio_dir
        )

    # --------------------------------
    # 5. Download slides
    # --------------------------------

    print("\n[5/7] Downloading slides...")

    slide_count = 0

    for url in slides:

        result = download_file(
            url,
            slides_dir
        )

        if result:
            slide_count += 1

    print(
        "Slides downloaded:",
        slide_count
    )

    # --------------------------------
    # 6. Extract annotations
    # --------------------------------

    print("\n[6/7] Extracting annotations...")

    extracted = extract_annotations(
        meeting_id,
        meeting_dir,
        ANNOTATION_ZIP
    )

    print(
        "Annotation files extracted:",
        extracted
    )

    # --------------------------------
    # 7. Create transcript + summary
    # --------------------------------

    print("\n[7/7] Creating transcript and reference summary...")

    records = create_timestamped_transcript(
        meeting_id,
        meeting_dir
    )

    sections = create_reference_summary(
        meeting_id,
        meeting_dir
    )

    # --------------------------------
    # Verification
    # --------------------------------

    verification = {
        "meeting_id": meeting_id,
        "video": os.path.exists(video_dir),
        "audio": os.path.exists(audio_dir),
        "slides": os.path.exists(slides_dir),
        "timestamped_transcript": len(records) > 0,
        "reference_summary": True,
        "dialogue_act_annotations": True,
        "timestamped_words": len(records),
        "abstract_sentences": len(
            sections["ABSTRACT"]
        ),
        "action_sentences": len(
            sections["ACTIONS"]
        ),
        "decision_sentences": len(
            sections["DECISIONS"]
        ),
        "problem_sentences": len(
            sections["PROBLEMS"]
        ),
        "status": "READY_FOR_PIPELINE",
        "verified_on": datetime.now().isoformat()
    }

    verification_path = os.path.join(
        meeting_dir,
        f"{meeting_id}_verification.json"
    )

    with open(
        verification_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            verification,
            f,
            indent=2
        )

    print("\n" + "=" * 60)
    print(f"{meeting_id} READY")
    print("=" * 60)

    print(
        json.dumps(
            verification,
            indent=2
        )
    )

    return verification

In [52]:
result_c = prepare_ami_meeting("ES2004c")

Preparing ES2004c

[1/7] Getting AMI download script...
Videos: 7
Audio: 1
Slides: 127

[2/7] Selecting accessible video...
Testing video: ES2004c.PreferredOverview.avi
  ✗ 403
Testing video: ES2004c.Closeup1.avi
  ✓ Accessible
Selected: ES2004c.Closeup1.avi

[3/7] Downloading video...
Downloading: ES2004c.Closeup1.avi
  ✓ 52.51 MB

[4/7] Downloading audio...
Downloading: ES2004c.Mix-Headset.wav
  ✓ 71.24 MB

[5/7] Downloading slides...
Downloading: ES2004c.1034.05__1046.80.txt
  ✓ 0.00 MB
Downloading: ES2004c.1046.80__1057.08.txt
  ✓ 0.00 MB
Downloading: ES2004c.105.87__116.47.txt
  ✓ 0.00 MB
Downloading: ES2004c.1057.08__1061.11.txt
  ✓ 0.00 MB
Downloading: ES2004c.1061.11__1064.65.txt
  ✓ 0.00 MB
Downloading: ES2004c.1064.65__1123.78.txt
  ✓ 0.00 MB
Downloading: ES2004c.1123.78__1136.26.txt
  ✓ 0.00 MB
Downloading: ES2004c.1136.26__1153.78.txt
  ✓ 0.00 MB
Downloading: ES2004c.1153.78__1172.48.txt
  ✓ 0.00 MB
Downloading: ES2004c.116.47__139.94.txt
  ✓ 0.00 MB
Downloading: ES2004c.11

In [53]:
remaining_meetings = [
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

print("Remaining meetings:")
for m in remaining_meetings:
    print(m)

print("\nTotal:", len(remaining_meetings))

Remaining meetings:
ES2004d
ES2005a
ES2005b
ES2005c
ES2006a
ES2006b
ES2008a

Total: 7


In [54]:
for meeting_id in remaining_meetings:

    try:
        script = get_ami_download_script(meeting_id)

        videos, audios, slides = extract_urls(
            script,
            meeting_id
        )

        print(
            f"{meeting_id}: "
            f"Videos={len(videos)}, "
            f"Audio={len(audios)}, "
            f"Slides={len(slides)}"
        )

    except Exception as e:
        print(f"{meeting_id}: ERROR -> {e}")

ES2004d: Videos=7, Audio=1, Slides=0
ES2005a: Videos=7, Audio=1, Slides=25
ES2005b: Videos=7, Audio=1, Slides=208
ES2005c: Videos=7, Audio=1, Slides=173
ES2006a: Videos=7, Audio=1, Slides=132
ES2006b: Videos=7, Audio=1, Slides=128
ES2008a: Videos=7, Audio=1, Slides=65


In [55]:
meeting_id = "ES2004d"

script_d = get_ami_download_script(meeting_id)

print("Script length:", len(script_d))

print("\nLines containing slides:")
slide_lines = [
    line for line in script_d.splitlines()
    if "/slides/" in line
]

print("Number of slide lines:", len(slide_lines))

for line in slide_lines[:20]:
    print(line)

Script length: 1390

Lines containing slides:
Number of slide lines: 0


In [56]:
for meeting_id in remaining_meetings:
    print("\n\n")
    result = prepare_ami_meeting(meeting_id)




Preparing ES2004d

[1/7] Getting AMI download script...
Videos: 7
Audio: 1
Slides: 0

[2/7] Selecting accessible video...
Testing video: ES2004d.PreferredOverview.avi
  ✗ 403
Testing video: ES2004d.Closeup1.avi
  ✓ Accessible
Selected: ES2004d.Closeup1.avi

[3/7] Downloading video...
Downloading: ES2004d.Closeup1.avi
  ✓ 50.26 MB

[4/7] Downloading audio...
Downloading: ES2004d.Mix-Headset.wav
  ✓ 67.82 MB

[5/7] Downloading slides...
Slides downloaded: 0

[6/7] Extracting annotations...
Annotation files extracted: 19

[7/7] Creating transcript and reference summary...

ES2004d READY
{
  "meeting_id": "ES2004d",
  "video": true,
  "audio": true,
  "slides": true,
  "timestamped_transcript": true,
  "reference_summary": true,
  "dialogue_act_annotations": true,
  "timestamped_words": 7337,
  "abstract_sentences": 11,
  "action_sentences": 2,
  "decision_sentences": 6,
  "problem_sentences": 2,
  "status": "READY_FOR_PIPELINE",
  "verified_on": "2026-08-26T08:15:58.985277"
}



Prepar

In [57]:
import os
import json

all_meetings = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

for meeting_id in all_meetings:

    path = os.path.join(
        AMI_DIR,
        meeting_id,
        f"{meeting_id}_verification.json"
    )

    if os.path.exists(path):

        with open(path, "r", encoding="utf-8") as f:
            v = json.load(f)

        print(
            f"{meeting_id}: "
            f"{v['status']} | "
            f"words={v['timestamped_words']} | "
            f"abstract={v['abstract_sentences']} | "
            f"actions={v['action_sentences']} | "
            f"decisions={v['decision_sentences']} | "
            f"problems={v['problem_sentences']}"
        )

    else:
        print(f"{meeting_id}: VERIFICATION FILE MISSING")


KeyError: 'timestamped_words'

In [58]:
import os
import json

all_meetings = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

for meeting_id in all_meetings:

    path = os.path.join(
        AMI_DIR,
        meeting_id,
        f"{meeting_id}_verification.json"
    )

    if not os.path.exists(path):
        print(f"{meeting_id}: VERIFICATION FILE MISSING")
        continue

    with open(path, "r", encoding="utf-8") as f:
        v = json.load(f)

    words = v.get("timestamped_words", "N/A")
    abstract = v.get("abstract_sentences", "N/A")
    actions = v.get("action_sentences", "N/A")
    decisions = v.get("decision_sentences", "N/A")
    problems = v.get("problem_sentences", "N/A")

    print(
        f"{meeting_id}: "
        f"{v.get('status', 'UNKNOWN')} | "
        f"words={words} | "
        f"abstract={abstract} | "
        f"actions={actions} | "
        f"decisions={decisions} | "
        f"problems={problems}"
    )

ES2004a: READY_FOR_PIPELINE | words=N/A | abstract=7 | actions=1 | decisions=1 | problems=1
ES2004b: READY_FOR_PIPELINE | words=7681 | abstract=9 | actions=1 | decisions=4 | problems=1
ES2004c: READY_FOR_PIPELINE | words=7850 | abstract=9 | actions=0 | decisions=1 | problems=2
ES2004d: READY_FOR_PIPELINE | words=7337 | abstract=11 | actions=2 | decisions=6 | problems=2
ES2005a: READY_FOR_PIPELINE | words=891 | abstract=4 | actions=3 | decisions=1 | problems=1
ES2005b: READY_FOR_PIPELINE | words=7199 | abstract=9 | actions=1 | decisions=3 | problems=1
ES2005c: READY_FOR_PIPELINE | words=8003 | abstract=11 | actions=1 | decisions=5 | problems=1
ES2006a: READY_FOR_PIPELINE | words=3205 | abstract=7 | actions=4 | decisions=3 | problems=3
ES2006b: READY_FOR_PIPELINE | words=7220 | abstract=10 | actions=4 | decisions=0 | problems=1
ES2008a: READY_FOR_PIPELINE | words=2887 | abstract=6 | actions=5 | decisions=6 | problems=2


In [59]:
meeting_id = "ES2004a"

transcript_path = os.path.join(
    AMI_DIR,
    meeting_id,
    "transcript",
    f"{meeting_id}_transcript.txt"
)

reference_path = os.path.join(
    AMI_DIR,
    meeting_id,
    f"{meeting_id}_reference_summary.txt"
)

with open(transcript_path, "r", encoding="utf-8") as f:
    transcript = f.read()

with open(reference_path, "r", encoding="utf-8") as f:
    reference = f.read()

print("=" * 70)
print("INPUT — MEETING TRANSCRIPT")
print("=" * 70)
print(transcript[:5000])

print("\n\n")
print("=" * 70)
print("OUTPUT — REFERENCE MINUTES")
print("=" * 70)
print(reference)

INPUT — MEETING TRANSCRIPT
[0.37 - 0.95] A: Hmm
[0.95 - 1.53] A: hmm
[1.53 - 1.76] A: hmm
[1.76 - 1.76] A: .
[10.99 - 11.02] B: Are
[11.02 - 12.13] B: we
[12.13 - 12.29] B: we're
[12.29 - 12.42] B: not
[12.42 - 12.62] B: allowed
[12.62 - 12.70] B: to
[12.70 - 12.84] B: dim
[12.84 - 12.91] B: the
[12.91 - 13.18] B: lights
[13.18 - 13.31] B: so
[13.31 - 13.53] B: people
[13.53 - 13.71] B: can
[13.71 - 13.81] B: see
[13.81 - 13.96] B: that
[13.96 - 13.99] B: a
[13.99 - 14.15] B: bit
[14.15 - 14.53] B: better
[14.53 - 14.53] B: ?
[17.88 - 18.15] A: Yeah
[18.15 - 18.15] A: .
[18.87 - 19.70] B: Okay
[19.70 - 19.70] B: ,
[19.70 - 19.99] B: that's
[19.99 - 20.29] B: fine
[20.29 - 20.29] B: .
[22.37 - 22.50] B: Am
[22.50 - 22.56] B: I
[22.56 - 22.78] B: supposed
[22.78 - 22.84] B: to
[22.84 - 22.90] B: be
[22.90 - 23.28] B: standing
[23.28 - 23.44] B: up
[23.44 - 23.81] B: there
[23.81 - 23.81] B: ?
[25.15 - 25.23] D: So
[25.18 - 25.60] B: Okay
[25.23 - 25.33] D: we've
[25.33 - 25.46] D: got
[2

In [60]:
# Create a clean input-output dataset containing the transcript and reference MoM for all 10 meetings.

import os
import json
import pandas as pd

all_meetings = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

dataset = []

for meeting_id in all_meetings:

    meeting_dir = os.path.join(
        AMI_DIR,
        meeting_id
    )

    transcript_path = os.path.join(
        meeting_dir,
        "transcript",
        f"{meeting_id}_transcript.txt"
    )

    reference_path = os.path.join(
        meeting_dir,
        f"{meeting_id}_reference_summary.txt"
    )

    if not os.path.exists(transcript_path):
        print(f"⚠ Transcript missing: {meeting_id}")
        continue

    if not os.path.exists(reference_path):
        print(f"⚠ Reference summary missing: {meeting_id}")
        continue

    with open(
        transcript_path,
        "r",
        encoding="utf-8"
    ) as f:
        transcript = f.read()

    with open(
        reference_path,
        "r",
        encoding="utf-8"
    ) as f:
        reference_summary = f.read()

    dataset.append({
        "meeting_id": meeting_id,
        "transcript": transcript,
        "reference_summary": reference_summary
    })


baseline_df = pd.DataFrame(dataset)

print("Meetings successfully loaded:", len(baseline_df))
print("\nColumns:")
print(baseline_df.columns.tolist())

print("\nDataset shape:")
print(baseline_df.shape)

print("\nMeeting IDs:")
print(baseline_df["meeting_id"].tolist())

Meetings successfully loaded: 10

Columns:
['meeting_id', 'transcript', 'reference_summary']

Dataset shape:
(10, 3)

Meeting IDs:
['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']


In [62]:
# Save the 10-meeting transcript-to-MoM dataset in the project's processed-data folder.

processed_dir = os.path.join(
    AMI_DIR,
    "..",
    "processed"
)

processed_dir = os.path.abspath(processed_dir)

os.makedirs(
    processed_dir,
    exist_ok=True
)

baseline_path = os.path.join(
    processed_dir,
    "ami_baseline_dataset.csv"
)

baseline_df.to_csv(
    baseline_path,
    index=False,
    encoding="utf-8"
)

print("Baseline dataset saved:")
print(baseline_path)

print("\nFile size:")
print(
    round(
        os.path.getsize(baseline_path) / (1024 * 1024),
        2
    ),
    "MB"
)

Baseline dataset saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/processed/ami_baseline_dataset.csv

File size:
1.42 MB


In [63]:
# Reload the saved dataset and verify that all 10 meeting input-output pairs are intact.

check_df = pd.read_csv(
    baseline_path,
    encoding="utf-8"
)

print("Reloaded successfully.")
print("Shape:", check_df.shape)
print("Meetings:", check_df["meeting_id"].tolist())

print("\nMissing values:")
print(check_df.isnull().sum())

Reloaded successfully.
Shape: (10, 3)
Meetings: ['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']

Missing values:
meeting_id           0
transcript           0
reference_summary    0
dtype: int64


In [64]:
# Calculate transcript and reference-summary lengths to determine the required preprocessing strategy.

baseline_df["transcript_words"] = baseline_df["transcript"].apply(
    lambda x: len(str(x).split())
)

baseline_df["reference_words"] = baseline_df["reference_summary"].apply(
    lambda x: len(str(x).split())
)

print(
    baseline_df[
        [
            "meeting_id",
            "transcript_words",
            "reference_words"
        ]
    ].to_string(index=False)
)

print("\nAverage transcript words:",
      round(baseline_df["transcript_words"].mean()))

print("Average reference words:",
      round(baseline_df["reference_words"].mean()))

print("Maximum transcript words:",
      baseline_df["transcript_words"].max())

print("Maximum reference words:",
      baseline_df["reference_words"].max())

meeting_id  transcript_words  reference_words
   ES2004a             15675              178
   ES2004b             38405              319
   ES2004c             39250              275
   ES2004d             36685              347
   ES2005a              4455              128
   ES2005b             35995              266
   ES2005c             40015              312
   ES2006a             16025              274
   ES2006b             36100              282
   ES2008a             14435              304

Average transcript words: 27704
Average reference words: 268
Maximum transcript words: 40015
Maximum reference words: 347


In [65]:
# Check the installed NLP libraries and runtime environment before selecting the baseline model.

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

try:
    import transformers
    print("Transformers:", transformers.__version__)
except ImportError:
    print("Transformers: NOT INSTALLED")

try:
    import datasets
    print("Datasets:", datasets.__version__)
except ImportError:
    print("Datasets: NOT INSTALLED")

try:
    import evaluate
    print("Evaluate:", evaluate.__version__)
except ImportError:
    print("Evaluate: NOT INSTALLED")

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cpu
CUDA available: False
Transformers: 5.15.0
Datasets: 4.0.0
Evaluate: NOT INSTALLED


In [1]:
# Verify that the Colab runtime has successfully switched from CPU to GPU.

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU not detected.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [2]:
# Check the available GPU memory before loading the baseline summarization model.

!nvidia-smi

Wed Aug 26 08:45:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Load the FLAN-T5-base tokenizer and model onto the available Tesla T4 GPU.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Model:", MODEL_NAME)
print("Device:", device)
print("Model loaded successfully.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-base
Device: cuda
Model loaded successfully.


In [4]:
# Test FLAN-T5-base on a small portion of the ES2004a transcript before building the full summarization pipeline.

sample_transcript = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "transcript"
].iloc[0]

sample_text = sample_transcript[:4000]

prompt = f"""
Summarize the following meeting transcript in concise meeting-minutes style.

Transcript:
{sample_text}

Summary:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4,
        early_stopping=True
    )

generated_summary = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("GENERATED BASELINE SUMMARY")
print("=" * 70)
print(generated_summary)

NameError: name 'baseline_df' is not defined

In [5]:
# Restore the project paths and reload the previously saved 10-meeting baseline dataset.

import os
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

AMI_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami"
)

baseline_path = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "ami_baseline_dataset.csv"
)

baseline_df = pd.read_csv(
    baseline_path,
    encoding="utf-8"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("AMI directory:", AMI_DIR)
print("Baseline dataset:", baseline_path)
print("Dataset shape:", baseline_df.shape)
print("Device:", device)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed/ami_baseline_dataset.csv'

In [6]:
# Locate the previously saved baseline CSV anywhere inside the project folder.

import os

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

matches = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if file == "ami_baseline_dataset.csv":
            matches.append(os.path.join(root, file))

print("Baseline dataset files found:", len(matches))

for path in matches:
    print(path)

Baseline dataset files found: 0


In [7]:
# Rebuild the 10-meeting baseline dataset directly from the prepared AMI folders.

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

AMI_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami"
)

all_meetings = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

dataset = []

for meeting_id in all_meetings:

    meeting_dir = os.path.join(
        AMI_DIR,
        meeting_id
    )

    transcript_path = os.path.join(
        meeting_dir,
        "transcript",
        f"{meeting_id}_transcript.txt"
    )

    reference_path = os.path.join(
        meeting_dir,
        f"{meeting_id}_reference_summary.txt"
    )

    if not os.path.exists(transcript_path):
        print(f"⚠ Transcript missing: {meeting_id}")
        continue

    if not os.path.exists(reference_path):
        print(f"⚠ Reference missing: {meeting_id}")
        continue

    with open(
        transcript_path,
        "r",
        encoding="utf-8"
    ) as f:
        transcript = f.read()

    with open(
        reference_path,
        "r",
        encoding="utf-8"
    ) as f:
        reference_summary = f.read()

    dataset.append({
        "meeting_id": meeting_id,
        "transcript": transcript,
        "reference_summary": reference_summary
    })

baseline_df = pd.DataFrame(dataset)

print("Meetings loaded:", len(baseline_df))
print("Shape:", baseline_df.shape)
print("Meetings:", baseline_df["meeting_id"].tolist())

⚠ Transcript missing: ES2004a
⚠ Transcript missing: ES2004b
⚠ Transcript missing: ES2004c
⚠ Transcript missing: ES2004d
⚠ Transcript missing: ES2005a
⚠ Transcript missing: ES2005b
⚠ Transcript missing: ES2005c
⚠ Transcript missing: ES2006a
⚠ Transcript missing: ES2006b
⚠ Transcript missing: ES2008a
Meetings loaded: 0
Shape: (0, 0)


KeyError: 'meeting_id'

In [8]:
# Locate the actual transcript and reference-summary files inside the prepared AMI dataset.

import os

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
AMI_DIR = os.path.join(PROJECT_DIR, "data", "raw", "ami")

for meeting_id in [
    "ES2004a", "ES2004b", "ES2004c", "ES2004d",
    "ES2005a", "ES2005b", "ES2005c",
    "ES2006a", "ES2006b", "ES2008a"
]:

    meeting_dir = os.path.join(AMI_DIR, meeting_id)

    print(f"\n===== {meeting_id} =====")

    if not os.path.exists(meeting_dir):
        print("Meeting directory NOT FOUND")
        continue

    for root, dirs, files in os.walk(meeting_dir):

        for file in files:

            if (
                "transcript" in file.lower()
                or "summary" in file.lower()
                or "summ" in file.lower()
            ):
                print(
                    os.path.join(
                        root,
                        file
                    )
                )


===== ES2004a =====
Meeting directory NOT FOUND

===== ES2004b =====
Meeting directory NOT FOUND

===== ES2004c =====
Meeting directory NOT FOUND

===== ES2004d =====
Meeting directory NOT FOUND

===== ES2005a =====
Meeting directory NOT FOUND

===== ES2005b =====
Meeting directory NOT FOUND

===== ES2005c =====
Meeting directory NOT FOUND

===== ES2006a =====
Meeting directory NOT FOUND

===== ES2006b =====
Meeting directory NOT FOUND

===== ES2008a =====
Meeting directory NOT FOUND


In [9]:
# Locate the actual MoM_Project folder in Google Drive after the Colab runtime restart.

import os

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive"
]

matches = []

for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    for root, dirs, files in os.walk(search_root):

        # Avoid unnecessarily traversing hidden/system folders
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        if os.path.basename(root) == "MoM_Project":
            matches.append(root)

print("MoM_Project folders found:", len(matches))

for path in matches:
    print(path)

MoM_Project folders found: 0


In [10]:
# Mount Google Drive so the previously downloaded AMI dataset becomes accessible again.

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
# Confirm that Google Drive is mounted and inspect the top-level folders.

import os

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

print("\nMyDrive folders:")
for item in os.listdir("/content/drive/MyDrive"):
    print(" -", item)

Drive mounted: True

MyDrive folders:
 - Sunaina_doc
 - Irfan
 - Sunaina Resume
 - PES
 - Colab Notebooks
 - recent year paper pleae.gsheet
 - MTechIndProj
 - Untitled


In [12]:
# Locate the MoM_Project folder and confirm the existing AMI data is still present.

import os

mtech_dir = "/content/drive/MyDrive/MTechIndProj"

print("MTechIndProj contents:")

for item in os.listdir(mtech_dir):
    print(" -", item)

print("\nSearching for MoM_Project...")

matches = []

for root, dirs, files in os.walk(mtech_dir):
    if os.path.basename(root) == "MoM_Project":
        matches.append(root)

print("\nMoM_Project folders found:", len(matches))

for path in matches:
    print(path)

MTechIndProj contents:
 - MoM_Project

Searching for MoM_Project...

MoM_Project folders found: 1
/content/drive/MyDrive/MTechIndProj/MoM_Project


In [13]:
# Confirm the existing AMI dataset and locate the generated transcript/reference files after remounting Drive.

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
AMI_DIR = os.path.join(PROJECT_DIR, "data", "raw", "ami")

print("AMI directory exists:", os.path.exists(AMI_DIR))
print("AMI path:", AMI_DIR)

for meeting_id in [
    "ES2004a", "ES2004b", "ES2004c", "ES2004d",
    "ES2005a", "ES2005b", "ES2005c",
    "ES2006a", "ES2006b", "ES2008a"
]:

    meeting_dir = os.path.join(AMI_DIR, meeting_id)

    print(f"\n===== {meeting_id} =====")

    if not os.path.exists(meeting_dir):
        print("❌ Meeting folder missing")
        continue

    found = []

    for root, dirs, files in os.walk(meeting_dir):
        for file in files:
            if (
                "transcript" in file.lower()
                or "reference_summary" in file.lower()
            ):
                found.append(os.path.join(root, file))

    print("Files found:", len(found))

    for path in found:
        print(" -", path)

AMI directory exists: True
AMI path: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami

===== ES2004a =====
Files found: 3
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_reference_summary.txt
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/transcript/ES2004a_timestamped_transcript.json
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/transcript/ES2004a_transcript.txt

===== ES2004b =====
Files found: 3
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/ES2004b_reference_summary.txt
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/transcript/ES2004b_timestamped_transcript.json
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004b/transcript/ES2004b_transcript.txt

===== ES2004c =====
Files found: 3
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004c/ES2004c_reference_summary.txt
 - /content/drive/MyDrive/MTechIndProj

In [14]:
# Rebuild the 10-meeting baseline dataset using the verified AMI file structure.

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
AMI_DIR = os.path.join(PROJECT_DIR, "data", "raw", "ami")

MEETING_IDS = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

records = []

for meeting_id in MEETING_IDS:

    meeting_dir = os.path.join(AMI_DIR, meeting_id)

    transcript_path = os.path.join(
        meeting_dir,
        "transcript",
        f"{meeting_id}_transcript.txt"
    )

    reference_path = os.path.join(
        meeting_dir,
        f"{meeting_id}_reference_summary.txt"
    )

    with open(transcript_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    with open(reference_path, "r", encoding="utf-8") as f:
        reference_summary = f.read()

    records.append({
        "meeting_id": meeting_id,
        "transcript": transcript,
        "reference_summary": reference_summary
    })

baseline_df = pd.DataFrame(records)

print("Meetings successfully loaded:", len(baseline_df))
print("Shape:", baseline_df.shape)
print("\nMeeting IDs:")
print(baseline_df["meeting_id"].tolist())

Meetings successfully loaded: 10
Shape: (10, 3)

Meeting IDs:
['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']


In [15]:
# Save the reconstructed baseline dataset to the processed-data directory and verify it.

PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)

baseline_path = os.path.join(
    PROCESSED_DIR,
    "ami_baseline_dataset.csv"
)

baseline_df.to_csv(
    baseline_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", baseline_path)
print("File exists:", os.path.exists(baseline_path))
print(
    "File size:",
    round(os.path.getsize(baseline_path) / (1024 * 1024), 2),
    "MB"
)

Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed/ami_baseline_dataset.csv
File exists: True
File size: 1.42 MB


In [16]:
# Reload the saved CSV to make sure the dataset survives a future runtime restart.

test_df = pd.read_csv(
    baseline_path,
    encoding="utf-8"
)

print("Reload successful.")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
print("Meetings:", test_df["meeting_id"].tolist())
print("\nMissing values:")
print(test_df.isnull().sum())

Reload successful.
Shape: (10, 3)
Columns: ['meeting_id', 'transcript', 'reference_summary']
Meetings: ['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']

Missing values:
meeting_id           0
transcript           0
reference_summary    0
dtype: int64


In [17]:
# Reload FLAN-T5-base onto the Tesla T4 after the Colab runtime restart.

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Model loaded successfully.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model: google/flan-t5-base
Device: cuda
GPU: Tesla T4
Model loaded successfully.


In [18]:
# Test the baseline model on a small ES2004a transcript segment before processing full meetings.

sample_transcript = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "transcript"
].iloc[0]

sample_text = sample_transcript[:4000]

prompt = f"""
Summarize the following meeting transcript in concise meeting-minutes style.

Transcript:
{sample_text}

Summary:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4
    )

generated_summary = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("GENERATED BASELINE SUMMARY")
print("=" * 70)
print(generated_summary)

GENERATED BASELINE SUMMARY
[25.15 - 25.23] A: . [10.99 - 11.02] B: Are [12.13 - 12.29] B: we [12.29 - 12.42] B: not [12.42 - 12.62] B: allowed [12.62 - 12.70] B: to [12.70 - 12.84] B: dim [12.84 - 12.91] B: the [12.91 - 13.18] B: lights [13.18 - 13.31] B: so [13.31 - 13.53] B: people [13.53 - 13.71] B: can [13.71 - 13.81] B: can


In [19]:
# Create clean conversational transcripts for baseline summarization while preserving the original timestamped data.

import re

def clean_timestamped_transcript(text):
    """
    Remove [start - end] timestamps while retaining speaker labels and words.
    """

    text = re.sub(
        r"\[\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\]\s*",
        "",
        text
    )

    return text.strip()


baseline_df["clean_transcript"] = baseline_df["transcript"].apply(
    clean_timestamped_transcript
)

print("Clean transcript created.")
print("\nOriginal:")
print(baseline_df.loc[0, "transcript"][:500])

print("\nClean:")
print(baseline_df.loc[0, "clean_transcript"][:500])

Clean transcript created.

Original:
[0.37 - 0.95] A: Hmm
[0.95 - 1.53] A: hmm
[1.53 - 1.76] A: hmm
[1.76 - 1.76] A: .
[10.99 - 11.02] B: Are
[11.02 - 12.13] B: we
[12.13 - 12.29] B: we're
[12.29 - 12.42] B: not
[12.42 - 12.62] B: allowed
[12.62 - 12.70] B: to
[12.70 - 12.84] B: dim
[12.84 - 12.91] B: the
[12.91 - 13.18] B: lights
[13.18 - 13.31] B: so
[13.31 - 13.53] B: people
[13.53 - 13.71] B: can
[13.71 - 13.81] B: see
[13.81 - 13.96] B: that
[13.96 - 13.99] B: a
[13.99 - 14.15] B: bit
[14.15 - 14.53] B: better
[14.53 - 14.53] 

Clean:
A: Hmm
A: hmm
A: hmm
A: .
B: Are
B: we
B: we're
B: not
B: allowed
B: to
B: dim
B: the
B: lights
B: so
B: people
B: can
B: see
B: that
B: a
B: bit
B: better
B: ?
A: Yeah
A: .
B: Okay
B: ,
B: that's
B: fine
B: .
B: Am
B: I
B: supposed
B: to
B: be
B: standing
B: up
B: there
B: ?
D: So
B: Okay
D: we've
D: got
D: both
B: .
D: of
D: these
D: clipped
D: on
D: ?
D: She
D: gonna
D: answer
D: me
B: Yeah
D: or
D: not
B: ,
B: I've
B: got
D: ?
D: Right
D: ,
D: bo

In [20]:
# Compare transcript sizes before and after timestamp removal.

baseline_df["timestamped_words"] = baseline_df["transcript"].str.split().str.len()
baseline_df["clean_words"] = baseline_df["clean_transcript"].str.split().str.len()

print(
    baseline_df[
        ["meeting_id", "timestamped_words", "clean_words"]
    ].to_string(index=False)
)

meeting_id  timestamped_words  clean_words
   ES2004a              15675         6270
   ES2004b              38405        15362
   ES2004c              39250        15700
   ES2004d              36685        14674
   ES2005a               4455         1782
   ES2005b              35995        14398
   ES2005c              40015        16006
   ES2006a              16025         6410
   ES2006b              36100        14440
   ES2008a              14435         5774


In [21]:
# Save the baseline dataset containing both original timestamped and clean transcripts.

baseline_df.to_csv(
    baseline_path,
    index=False,
    encoding="utf-8"
)

print("Saved successfully:", baseline_path)

Saved successfully: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed/ami_baseline_dataset.csv


In [22]:
# Test FLAN-T5-base again using the cleaned ES2004a transcript without timestamps.

sample_text = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "clean_transcript"
].iloc[0][:4000]

prompt = f"""
Summarize the following meeting transcript in concise meeting-minutes style.
Focus on the main discussion, decisions, actions, and problems.

Transcript:
{sample_text}

Meeting minutes:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4
    )

generated_summary = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("BASELINE SUMMARY — CLEAN TRANSCRIPT")
print("=" * 70)
print(generated_summary)

BASELINE SUMMARY — CLEAN TRANSCRIPT
B: Hello, B: Hello, B: Hello, B: Hello, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B: Okay, B:


In [23]:
# Check how much of the ES2004a transcript is actually reaching FLAN-T5.

sample_text = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "clean_transcript"
].iloc[0][:4000]

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=False
)

print("Characters:", len(sample_text))
print("Tokens:", inputs["input_ids"].shape[1])
print("Model max position:", tokenizer.model_max_length)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1842 > 512). Running this sequence through the model will result in indexing errors


Characters: 4000
Tokens: 1842
Model max position: 512


In [24]:
# This cell checks the actual MoM_Project directory structure so we can use the existing folders correctly.

import os

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

print("=" * 70)
print("PROJECT DIRECTORY")
print("=" * 70)
print(PROJECT_DIR)
print("Exists:", os.path.exists(PROJECT_DIR))

if os.path.exists(PROJECT_DIR):
    print("\nTop-level contents:")
    for item in sorted(os.listdir(PROJECT_DIR)):
        path = os.path.join(PROJECT_DIR, item)
        if os.path.isdir(path):
            print(f"📁 {item}/")
        else:
            print(f"📄 {item}")

    print("\n" + "=" * 70)
    print("FULL PROJECT STRUCTURE")
    print("=" * 70)

    for root, dirs, files in os.walk(PROJECT_DIR):
        # Skip very large/raw data contents for now
        dirs[:] = [d for d in dirs if d not in {"raw"}]

        level = root.replace(PROJECT_DIR, "").count(os.sep)
        indent = "    " * level
        folder_name = os.path.basename(root)

        print(f"{indent}📁 {folder_name}/")

        for file in sorted(files):
            print(f"{indent}    📄 {file}")

PROJECT DIRECTORY
/content/drive/MyDrive/MTechIndProj/MoM_Project
Exists: True

Top-level contents:
📁 00_setup/
📁 01_audio_vad/
📁 02_asr_diarization/
📁 03_dialogue_act/
📁 04_mom_generation/
📁 05_evidence_retrieval/
📁 06_verification/
📁 07_dashboard/
📁 08_evaluation/
📄 config.py
📁 data/
📁 evaluation_results/
📁 models/
📁 outputs/

FULL PROJECT STRUCTURE
📁 MoM_Project/
    📄 config.py
    📁 00_setup/
        📄 00_dataset_matching.ipynb
        📄 00_setup_and_project_structure.ipynb
    📁 01_audio_vad/
        📄 01_meetingbank_dataset.ipynb
    📁 02_asr_diarization/
    📁 03_dialogue_act/
    📁 04_mom_generation/
    📁 05_evidence_retrieval/
    📁 06_verification/
    📁 07_dashboard/
    📁 08_evaluation/
    📁 data/
        📁 audio/
        📁 vad/
        📁 transcripts/
        📁 dialogue_acts/
        📁 references/
            📄 0_reference_summary.txt
            📄 0_reference_transcript.txt
        📁 processed/
            📄 ami_baseline_dataset.csv
    📁 models/
        📁 asr/
        

In [25]:
# This cell checks the existing contents of the MoM generation phase before we add any new code.

import os

MOM_DIR = os.path.join(PROJECT_DIR, "04_mom_generation")

print("=" * 70)
print("04_MOM_GENERATION DIRECTORY")
print("=" * 70)
print("Path:", MOM_DIR)
print("Exists:", os.path.exists(MOM_DIR))

if os.path.exists(MOM_DIR):
    contents = os.listdir(MOM_DIR)

    if not contents:
        print("\nDirectory is currently empty.")
    else:
        print("\nExisting contents:")
        for item in sorted(contents):
            path = os.path.join(MOM_DIR, item)

            if os.path.isdir(path):
                print(f"📁 {item}/")
            else:
                print(f"📄 {item}")

04_MOM_GENERATION DIRECTORY
Path: /content/drive/MyDrive/MTechIndProj/MoM_Project/04_mom_generation
Exists: True

Directory is currently empty.


In [26]:
# This cell displays the existing project configuration so the MoM-generation phase uses the project's established paths and settings.

import os

CONFIG_PATH = "/content/drive/MyDrive/MTechIndProj/MoM_Project/config.py"

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)
print("Path:", CONFIG_PATH)
print("Exists:", os.path.exists(CONFIG_PATH))

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config_content = f.read()

    print("\n" + config_content)

PROJECT CONFIGURATION
Path: /content/drive/MyDrive/MTechIndProj/MoM_Project/config.py
Exists: True

{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"authorship_tag":"ABX9TyP7L3Gbqdn9g1wSYcCIBCMV"},"kernelspec":{"name":"python3","display_name":"Python 3"},"language_info":{"name":"python"}},"cells":[{"cell_type":"code","execution_count":null,"metadata":{"id":"BIASLfPWmcq4"},"outputs":[],"source":["import os\n","\n","PROJECT_DIR = \"/content/drive/MyDrive/MoM_Project\"\n","\n","# Data\n","RAW_DATA_DIR = os.path.join(PROJECT_DIR, \"data/raw\")\n","AUDIO_DIR = os.path.join(PROJECT_DIR, \"data/audio\")\n","VAD_DIR = os.path.join(PROJECT_DIR, \"data/vad\")\n","TRANSCRIPT_DIR = os.path.join(PROJECT_DIR, \"data/transcripts\")\n","DIALOGUE_ACT_DIR = os.path.join(PROJECT_DIR, \"data/dialogue_acts\")\n","REFERENCE_DIR = os.path.join(PROJECT_DIR, \"data/references\")\n","\n","# Models\n","ASR_MODEL_DIR = os.path.join(PROJECT_DIR, \"models/asr\")\n","DIARIZATION_MODEL_DIR = os.

In [27]:
# This cell replaces the incorrect notebook-formatted config.py with the project's actual Python configuration file.

import os

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
CONFIG_PATH = os.path.join(PROJECT_DIR, "config.py")

config_code = '''
import os
import torch

# Main project directory
PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

# Data directories
RAW_DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw")
AUDIO_DIR = os.path.join(PROJECT_DIR, "data", "audio")
VAD_DIR = os.path.join(PROJECT_DIR, "data", "vad")
TRANSCRIPT_DIR = os.path.join(PROJECT_DIR, "data", "transcripts")
DIALOGUE_ACT_DIR = os.path.join(PROJECT_DIR, "data", "dialogue_acts")
REFERENCE_DIR = os.path.join(PROJECT_DIR, "data", "references")
PROCESSED_DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")

# Model directories
ASR_MODEL_DIR = os.path.join(PROJECT_DIR, "models", "asr")
DIARIZATION_MODEL_DIR = os.path.join(PROJECT_DIR, "models", "diarization")
DIALOGUE_MODEL_DIR = os.path.join(PROJECT_DIR, "models", "dialogue_act")
LLM_MODEL_DIR = os.path.join(PROJECT_DIR, "models", "llm")

# Output directories
MOM_OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs", "mom")
EVIDENCE_OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs", "evidence")
VERIFIED_OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs", "verified")
PDF_OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs", "pdf")

# Evaluation
EVALUATION_DIR = os.path.join(PROJECT_DIR, "evaluation_results")

# Hardware
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
'''

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(config_code.strip() + "\n")

print("config.py corrected successfully.")
print("Path:", CONFIG_PATH)

config.py corrected successfully.
Path: /content/drive/MyDrive/MTechIndProj/MoM_Project/config.py


In [28]:
# This cell verifies that the corrected configuration points to the actual project and detects the available GPU.

import sys
import os

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import config

print("=" * 70)
print("CONFIGURATION VERIFICATION")
print("=" * 70)

print("Project:", config.PROJECT_DIR)
print("Project exists:", os.path.exists(config.PROJECT_DIR))
print("Device:", config.DEVICE)

print("\nImportant directories:")

paths_to_check = {
    "Processed data": config.PROCESSED_DATA_DIR,
    "LLM models": config.LLM_MODEL_DIR,
    "MoM output": config.MOM_OUTPUT_DIR,
    "Evidence output": config.EVIDENCE_OUTPUT_DIR,
    "Verified output": config.VERIFIED_OUTPUT_DIR,
    "Evaluation": config.EVALUATION_DIR,
}

for name, path in paths_to_check.items():
    print(f"{name:20} | {path} | Exists: {os.path.exists(path)}")

CONFIGURATION VERIFICATION
Project: /content/drive/MyDrive/MTechIndProj/MoM_Project
Project exists: True
Device: cuda

Important directories:
Processed data       | /content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed | Exists: True
LLM models           | /content/drive/MyDrive/MTechIndProj/MoM_Project/models/llm | Exists: True
MoM output           | /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom | Exists: True
Evidence output      | /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/evidence | Exists: True
Verified output      | /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/verified | Exists: True
Evaluation           | /content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results | Exists: True
